# B3 v3 Grid Search — BOVA11 Only

Definitive grid search using **rb3 DB** (`use_cdi_ivs=True`) with source tracking.

- **Market**: BOVA11 only (PETR4 conclusively negative across all v2 configs)
- **Data**: `b3_options_rb3.db` (1.4M rows, CDI-adjusted IVs)
- **Cash**: `risk_free` mode, `yield=0%` — pure strangle P&L, no B&H contamination
- **Grid**: 5×4×3×4×3×1 = 720 raw combos; ~630 after validity filter
- **Periods**: Train (2020-04-01 → 2023-12-31), Holdout (2024-01-01 → 2025-12-31)
- **Key metric**: `entry_db_pct` must appear in all outputs (synthetic fallback policy)
- **Min filter**: `n_chains ≥ 50` (not `n_trades ≥ 100`)
- **Speed**: Optimized — data pre-loaded, engine reused, source-mark merged (~2-3s/run)

In [ ]:
%matplotlib inline
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if (Path.cwd().name == "research") else Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
sys.path.insert(0, str(ROOT / "src"))
from grid_search import run_grid, enumerate_grid, GRID_DIMS

OUT_DIR = ROOT / "data" / "B3"
RESULTS_PATH = OUT_DIR / "v3_grid_search_results.parquet"

## 1. Grid Definition

In [ ]:
GRID = {
    "target_delta": [0.16, 0.20, 0.25, 0.30, 0.50],
    "dte_max": [21, 30, 45, 60],
    "profit_target_pct": [0.25, 0.50, 0.75],
    "manage_at_dte": [7, 10, 14, 21],
    "vix_filter": [
        "OFF",
        "rv_pct:80",
        "ivr:30",
    ],
    "defensive_leg_roll": [False],
}
combos = enumerate_grid(GRID)
print(f"Valid combos per (market, period): {len(combos)}")
print(f"Total backtests: {len(combos)} x 2 periods = {len(combos) * 2}")
print(f"~Est. runtime at 2s/run on 8 cores: {len(combos) * 2 * 2 / 8 / 60:.0f} minutes")

## 2. Run Configuration

In [ ]:
RUNS = [
    ("train", "bova11", "2020-04-01", "2023-12-31"),
    ("holdout", "bova11", "2024-01-01", "2025-12-31"),
]
for r in RUNS:
    print(r)

## 3. Execute Grid Search

Optimized: data and engine are cached per period. Estimated ~3 min on 8 cores.

In [ ]:
df = run_grid(GRID, RUNS, n_workers=-1, output_path=RESULTS_PATH)
print(f"Saved {len(df)} rows to {RESULTS_PATH}")

## 4. Load & Quick Summary

In [ ]:
df = pd.read_parquet(RESULTS_PATH)
print(df.shape)
df.head()

In [ ]:
print("Total rows:", len(df))
print("By period:")
print(df.groupby("period_label").size())
print("\nFailures:", (df["error"].notna()).sum())
print("Underpowered (n_chains < 50):", (df["n_chains"] < 50).sum())

clean = df[df["error"].isna()]
print("\n=== Strangle Sharpe (pure options, no B&H) ===")
print(clean.groupby("period_label")["sharpe"].describe()[["mean", "std", "min", "max"]])

print("\n=== Alpha Sharpe (residual vs B&H) ===")
print(clean.groupby("period_label")["alpha_sharpe"].describe()[["mean", "std", "min", "max"]])

print("\n=== Entry DB% (source tracking) ===")
print(clean.groupby("period_label")["entry_db_pct"].describe()[["mean", "std", "min", "max"]])

if (df["error"].notna()).sum() > 0:
    print(f"\nWARNING: {df['error'].notna().sum()} failed rows")
    failures = df[df["error"].notna()]
    for _, r in failures.iterrows():
        print(f"  [{r['period_label']}] vix={r['vix_filter']} d={r['target_delta']}"
              f" dte={r['dte_max']}: {str(r['error'])[:200]}")

## 5. Verification

In [ ]:
from grid_search import DTE_MIN_BY_DTE_MAX

invalid = df[
    (df["manage_at_dte"] >= df["dte_max"].map(DTE_MIN_BY_DTE_MAX)) & df["error"].isna()
]
assert len(invalid) == 0, f"{len(invalid)} rows violate manage_at_dte < dte_min!"
print(f"Check 1: validity filter — {len(invalid)} post-run violations (expect 0)")

print(f"Check 2: n_chains < 50 count: {(df['n_chains'] < 50).sum()}")

train = df.query("period_label=='train' and error.isna()", engine='python')
hold = df.query("period_label=='holdout' and error.isna()", engine='python')
j = train.merge(hold, on=GRID_DIMS, suffixes=("_t", "_h"))
print(f"Check 3: {len(j)}/{len(train)} train configs matched in holdout")

combos2 = enumerate_grid(GRID)
n_valid = df.groupby("period_label").size().unique()
print(f"Check 4: valid combos = {len(combos2)}, rows per period = {n_valid.tolist()}")
assert all(c == len(combos2) for c in n_valid), "diagonal size mismatch"

## 6. Source Tracking Diagnostics

Critical check: how much of the P&L comes from real DB data vs synthetic fallback?

In [ ]:
viable = df.query("error.isna() and n_chains >= 50", engine='python')
print(f"Viable configs (n_chains >= 50): {len(viable)} / {len(df[df['error'].isna()])}")
print()

print("=== entry_db_pct by target_delta (all viable configs) ===")
print(viable.groupby("target_delta")["entry_db_pct"].describe()[["mean", "50%", "min", "max"]])

print("\n=== entry_db_pct by target_delta × period ===")
pivot = viable.pivot_table(
    values="entry_db_pct", index="target_delta", columns="period_label", aggfunc="mean"
)
print(pivot.round(3))

print("\n=== Correlation: entry_db_pct vs sharpe ===")
for period in ["train", "holdout"]:
    sub = viable.query("period_label == @period")
    if len(sub) > 2:
        corr = sub["entry_db_pct"].corr(sub["sharpe"])
        print(f"  {period}: r = {corr:.3f}")

## 7. Heatmaps — Sharpe by target_delta x dte_max

In [ ]:
def heatmap(ax, sub, x, y, title, metric="sharpe", vmin=-1, vmax=1):
    pivot = sub.pivot_table(values=metric, index=y, columns=x, aggfunc="mean")
    im = ax.imshow(pivot.values, cmap="RdYlGn", vmin=vmin, vmax=vmax, aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index)
    ax.set_xlabel(x)
    ax.set_ylabel(y)
    ax.set_title(title)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            v = pivot.values[i, j]
            ax.text(
                j, i,
                f"{v:.2f}" if pd.notna(v) else "n/a",
                ha="center", va="center", color="black", fontsize=9,
            )
    return im

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, period in zip(axes, ["train", "holdout"]):
    sub = viable.query("period_label == @period")
    im = heatmap(ax, sub, "dte_max", "target_delta", f"BOVA11 — {period}")
fig.colorbar(im, ax=axes, label="Mean Strangle Sharpe", shrink=0.8)
plt.savefig(OUT_DIR / "v3_heatmap_delta_dte.png", dpi=120, bbox_inches="tight")
plt.show()

## 8. Heatmaps — Profit Target × Manage DTE

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, period in zip(axes, ["train", "holdout"]):
    sub = viable.query("period_label == @period")
    im = heatmap(ax, sub, "manage_at_dte", "profit_target_pct", f"BOVA11 — {period}")
fig.colorbar(im, ax=axes, label="Mean Strangle Sharpe", shrink=0.8)
plt.savefig(OUT_DIR / "v3_heatmap_profit_manage.png", dpi=120, bbox_inches="tight")
plt.show()

## 9. Scatter — n_chains vs Sharpe

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, period in zip(axes, ["train", "holdout"]):
    sub = viable.query("period_label == @period")
    sc = ax.scatter(
        sub["n_chains"], sub["sharpe"],
        c=sub["entry_db_pct"], cmap="coolwarm", vmin=0, vmax=1, s=18, alpha=0.7,
    )
    ax.axvline(50, ls="--", c="grey", lw=0.8, label="n_chains=50")
    ax.axhline(0, ls=":", c="black", lw=0.5)
    ax.set_xlabel("n_chains")
    ax.set_ylabel("Strangle Sharpe")
    ax.set_title(f"BOVA11 — {period}")
    ax.legend(loc="lower right")
fig.colorbar(sc, ax=axes, label="entry_db_pct", shrink=0.8)
plt.savefig(OUT_DIR / "v3_scatter_nchains_sharpe.png", dpi=120, bbox_inches="tight")
plt.show()

## 10. Vol Filter Impact

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, period in zip(axes, ["train", "holdout"]):
    sub = viable.query("period_label == @period")
    vf_order = ["OFF", "rv_pct:80", "ivr:30"]
    vf_data = [sub.query("vix_filter == @vf")["sharpe"].values for vf in vf_order]
    bp = ax.boxplot(vf_data, labels=vf_order, patch_artist=True)
    ax.axhline(0, ls=":", c="black", lw=0.5)
    ax.set_ylabel("Strangle Sharpe")
    ax.set_title(f"BOVA11 — {period}")
    ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig(OUT_DIR / "v3_vol_filter_boxplot.png", dpi=120, bbox_inches="tight")
plt.show()

## 11. Top-10 Configs by Sharpe

In [ ]:
COLS = [
    "sharpe", "alpha_sharpe", "alpha_vs_bh", "total_return", "calmar",
    "n_chains", "n_trades", "entry_db_pct", "win_rate",
] + GRID_DIMS

for period in ["train", "holdout"]:
    print(f"\n=== BOVA11 top-10 by {period} sharpe (n_chains >= 50) ===")
    sub = viable.query("period_label == @period")
    top = sub.nlargest(10, "sharpe")[COLS]
    print(top.to_string(index=False))

## 12. Train vs Holdout Scatter

In [ ]:
train_v = viable.query("period_label == 'train'")
hold_v = viable.query("period_label == 'holdout'")
j = train_v.merge(hold_v, on=GRID_DIMS, suffixes=("_t", "_h"))

fig, ax = plt.subplots(figsize=(10, 8))
sc = ax.scatter(
    j["sharpe_t"], j["sharpe_h"],
    c=j["entry_db_pct_t"], cmap="coolwarm", vmin=0, vmax=1,
    s=18, alpha=0.7,
)
lim = max(abs(j[["sharpe_t", "sharpe_h"]].values).max() if len(j) else 1.0, 1.0)
ax.plot([-lim, lim], [-lim, lim], ls="--", c="grey", label="y = x")
ax.axhline(0, ls=":", c="black", lw=0.5)
ax.axvline(0, ls=":", c="black", lw=0.5)
ax.set_xlabel("Sharpe (train)")
ax.set_ylabel("Sharpe (holdout)")
ax.set_title("BOVA11: train vs holdout (color = entry_db_pct)")
ax.legend(loc="upper left")
fig.colorbar(sc, label="entry_db_pct (train)", shrink=0.8)
plt.savefig(OUT_DIR / "v3_train_vs_holdout.png", dpi=120, bbox_inches="tight")
plt.show()

if len(j) > 2:
    corr = j["sharpe_t"].corr(j["sharpe_h"])
    print(f"Train-holdout Sharpe correlation: r = {corr:.3f}")
    print(f"Mean holdout Sharpe: {j['sharpe_h'].mean():.3f}")
    print(f"% holdout positive Sharpe: {(j['sharpe_h'] > 0).mean():.1%}")

## 13. DB-Entry Subset Analysis

Are the positive Sharpe configs driven by DB-priced entries or synthetic fallback?

In [ ]:
high_db = viable.query("entry_db_pct >= 0.70")
low_db = viable.query("entry_db_pct < 0.50")

for period in ["train", "holdout"]:
    print(f"\n=== {period.upper()} — DB entry analysis ===")
    for label, sub in [("high DB% (>=70%)", high_db.query("period_label == @period")),
                       ("low DB% (<50%)", low_db.query("period_label == @period"))]:
        if len(sub) == 0:
            print(f"  {label}: no configs")
            continue
        print(f"  {label}: n={len(sub)}, sharpe={sub['sharpe'].mean():.2f} +/- {sub['sharpe'].std():.2f},"
              f" tot_ret={sub['total_return'].mean():.3f},"
              f" win={sub['win_rate'].mean():.1%}")

print("\n=== Positive Sharpe configs: entry_db_pct distribution ===")
pos_sharpe = viable.query("sharpe > 0")
neg_sharpe = viable.query("sharpe <= 0")
for period in ["train", "holdout"]:
    p = pos_sharpe.query("period_label == @period")
    n = neg_sharpe.query("period_label == @period")
    if len(p) > 0:
        print(f"  {period} positive Sharpe (n={len(p)}): entry_db_pct mean={p['entry_db_pct'].mean():.3f}")
    if len(n) > 0:
        print(f"  {period} negative/sharpe (n={len(n)}): entry_db_pct mean={n['entry_db_pct'].mean():.3f}")

## 14. Defensive Leg Roll

All v3 configs use `defensive_leg_roll=False` (proven harmful on B3). This section
is a placeholder for a follow-up comparison if needed.

In [ ]:
print("Note: all v3 configs have defensive_leg_roll=False (B3 analysis showed 68% worse P&L)")

## 15. Conclusions

Fill in after running:
- **Top-cluster signal**: Are top-10 train configs concentrated or scattered?
- **Holdout robustness**: Do top configs cluster on/above the y=x diagonal?
- **Sharpe sign**: Is any positive strangle Sharpe achievable on BOVA11?
- **Alpha vs B&H**: Does the strangle overlay add risk-adjusted value?
- **Source tracking**: Is the positive ridge DB-backed or synthetic-driven?
- **Vol filter**: Which B3-native mode (rv_pct / ivr) is most effective?
- **Straddle check**: Does target_delta=0.50 confirm the structural loss we expect?
